In [2]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
SYSTEM_PROMPT = """
You are generating synthetic training data for a GAD-7 survey chatbot.

Your job is to produce one realistic multi-turn conversation and its final normalized answers.

Requirements:
- The conversation must cover all 7 GAD-7 questions exactly once, in order.
- The assistant may ask clarification follow-ups when the user's answer is vague or ambiguous.
- The user should sound natural and human, not robotic.
- The user should sometimes answer vaguely, indirectly, or with words instead of numbers.
- The assistant must never decide an answer on behalf of the user.
- The assistant must not skip any question.
- The assistant must not merge questions together.
- The assistant may restate the scale only to clarify ambiguity.
- The final normalized answer for each question must be a single string value: "0", "1", "2", or "3".

GAD-7 question order:
q1: Over the last two weeks, how often have you been bothered by feeling nervous, anxious, or on edge?
q2: Over the last two weeks, how often have you been bothered by not being able to stop or control worrying?
q3: Over the last two weeks, how often have you been bothered by worrying too much about different things?
q4: Over the last two weeks, how often have you been bothered by trouble relaxing?
q5: Over the last two weeks, how often have you been bothered by being so restless that it is hard to sit still?
q6: Over the last two weeks, how often have you been bothered by becoming easily annoyed or irritable?
q7: Over the last two weeks, how often have you been bothered by feeling afraid, as if something awful might happen?

Scale mapping:
0 = not at all
1 = several days
2 = more than half the days
3 = nearly every day

Output rules:
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q7 key.
- expected_output must contain exactly q1 through q7.
- conversation must be consistent with expected_output.

Return JSON in exactly this shape:
{
  "conversation": [
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."}
  ],
  "expected_output": {
    "q1": "0",
    "q2": "0",
    "q3": "0",
    "q4": "0",
    "q5": "0",
    "q6": "0",
    "q7": "0"
  }
}
"""
MODEL = "gpt-4o-mini"

In [4]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

In [5]:
client = OpenAI()

completion = client.responses.create(
    model=MODEL,
    input=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                "Generate one realistic GAD-7 conversation. "
                "The user should give at least 2 vague answers that require clarification. "
                "Do not skip any of the 7 questions."
            )
        }
    ]
)

print(completion.output[0].content[0].text)

{
  "conversation": [
    {"role": "assistant", "content": "Hi there! I'm here to ask you some questions for the GAD-7 survey. Let's start with the first one. Over the last two weeks, how often have you been bothered by feeling nervous, anxious, or on edge?"},
    {"role": "user", "content": "Hmm, I'd say it hasn't been too much, but sometimes I do feel a bit tense."},
    {"role": "assistant", "content": "Just to clarify, would you say that's not at all, several days, more than half the days, or nearly every day?"},
    {"role": "user", "content": "Probably just several days."},
    {"role": "assistant", "content": "Got it! So for question one, that's a 1. Moving on to question two. Over the last two weeks, how often have you been bothered by not being able to stop or control worrying?"}, 
    {"role": "user", "content": "I guess I worry occasionally, but it's not all-consuming."},
    {"role": "assistant", "content": "Would you say that means not at all, several days, more than half 

In [6]:
def extra_checks(sample):
    convo_text = " ".join(
        m["content"].lower()
        for m in sample["conversation"]
        if m["role"] == "assistant"
    )

    checks = {
        "q1": ["nervous", "anxious", "on edge"],
        "q2": ["stop or control worrying"],
        "q3": ["worrying too much", "different things"],
        "q4": ["trouble relaxing"],
        "q5": ["restless", "hard to sit still"],
        "q6": ["annoyed", "irritable"],
        "q7": ["afraid", "something awful might happen"]
    }

    for q, keywords in checks.items():
        for keyword in keywords:
            if keyword not in convo_text:
                return False, f"Missing {q} keyword: {keyword}"

    expected_keys = {"q1", "q2", "q3", "q4", "q5", "q6", "q7"}
    if set(sample["expected_output"].keys()) != expected_keys:
        return False, "expected_output must contain exactly q1-q7"

    for k, v in sample["expected_output"].items():
        if v not in {"0", "1", "2", "3"}:
            return False, f"{k} must be one of 0,1,2,3"

    return True, "OK"

In [7]:
import json

def generate_sample():
    completion = client.responses.create(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": "Generate one realistic GAD-7 conversation. Include at least 2 vague answers that require clarification."
            }
        ]
    )
    text = completion.output[0].content[0].text
    return json.loads(text)

sample = generate_sample()

valid, message = extra_checks(sample)

print("Validation:", "PASS" if valid else "FAIL", "-", message)

print("\n--- Conversation ---")
for msg in sample["conversation"]:
    print(f"{msg['role'].upper()}: {msg['content']}")

print("\n--- Expected Output ---")
print(json.dumps(sample["expected_output"], indent=2))

Validation: PASS - OK

--- Conversation ---
ASSISTANT: Hi there! I’d like to ask you a few questions about how you've been feeling lately. Let's start with the first one. Over the last two weeks, how often have you been bothered by feeling nervous, anxious, or on edge?
USER: I guess I felt a little like that sometimes.
ASSISTANT: Got it! Just to clarify, would you say it was 'not at all', 'several days', 'more than half the days', or 'nearly every day'?
USER: Maybe several days.
ASSISTANT: Okay, thank you! Now, over the last two weeks, how often have you been bothered by not being able to stop or control worrying?
USER: Worrying, um, it happens every now and then.
ASSISTANT: I see. Just to help, would you say that was 'not at all', 'several days', 'more than half the days', or 'nearly every day'?
USER: Probably several days too.
ASSISTANT: Thanks for that! Now, how about worrying too much about different things? How often has that bothered you over the last two weeks?
USER: It can happ

In [8]:
import time

valid_samples = []
failed_samples = []

target_count = 10
attempts = 0
max_attempts = 30

while len(valid_samples) < target_count and attempts < max_attempts:
    attempts += 1
    try:
        sample = generate_sample()
        valid, message = extra_checks(sample)

        if valid:
            valid_samples.append(sample)
            print(f"Sample {attempts}: PASS ({len(valid_samples)}/{target_count})")
        else:
            failed_samples.append({"sample": sample, "reason": message})
            print(f"Sample {attempts}: FAIL - {message}")

        time.sleep(0.5)

    except Exception as e:
        print(f"Sample {attempts}: ERROR - {e}")

print(f"\nFinished. Kept {len(valid_samples)} valid samples out of {attempts} attempts.")

Sample 1: PASS (1/10)
Sample 2: PASS (2/10)
Sample 3: PASS (3/10)
Sample 4: PASS (4/10)
Sample 5: PASS (5/10)
Sample 6: PASS (6/10)
Sample 7: PASS (7/10)
Sample 8: PASS (8/10)
Sample 9: PASS (9/10)
Sample 10: PASS (10/10)

Finished. Kept 10 valid samples out of 10 attempts.


In [9]:
with open("gad_valid_dataset.json", "w") as f:
    json.dump(valid_samples, f, indent=2)

print("Saved valid samples to gad_valid_dataset.json")

Saved valid samples to gad_valid_dataset.json


In [10]:
print(json.dumps(valid_samples[0], indent=2))

{
  "conversation": [
    {
      "role": "assistant",
      "content": "Hi there! I'm here to help you with a few questions. Let's start. Over the last two weeks, how often have you been bothered by feeling nervous, anxious, or on edge?"
    },
    {
      "role": "user",
      "content": "I guess a bit, but not all the time."
    },
    {
      "role": "assistant",
      "content": "Just to clarify, would you say it's not at all, several days, more than half the days, or nearly every day?"
    },
    {
      "role": "user",
      "content": "Hmm, maybe several days."
    },
    {
      "role": "assistant",
      "content": "Got it! Now, over the last two weeks, how often have you been bothered by not being able to stop or control worrying?"
    },
    {
      "role": "user",
      "content": "Sometimes, but it varies."
    },
    {
      "role": "assistant",
      "content": "Could you indicate if that's not at all, several days, more than half the days, or nearly every day?"
    },


In [11]:
def compare_outputs(predicted, expected):
    per_question = {}
    correct = 0

    for k in expected:
        is_correct = predicted.get(k) == expected[k]
        per_question[k] = {
            "predicted": predicted.get(k),
            "expected": expected[k],
            "correct": is_correct
        }
        if is_correct:
            correct += 1

    accuracy = correct / len(expected)

    return {
        "accuracy": accuracy,
        "correct_count": correct,
        "total": len(expected),
        "per_question": per_question
    }